<details>
<summary><b>Info</b></summary>

**Last Execution:** 2026-07-25

| Package | Version |
|---------|---------|
| **nnsight** | **0.8** |
| Python | 3.12.13 |
| torch | 2.13.0+cu126 |
| transformers | 5.15.0 |

</details>


# Cross-Prompt Intervention

A single `trace()` can hold several `tracer.invoke(...)` blocks. Their inputs are combined into **one batched forward**, and each block's interventions see only *its* rows of every activation. Cross-prompt intervention means moving an activation out of one invoke and into another — the basis of techniques like activation patching, where a representation from a clean prompt is transferred into a corrupt one.

The key mechanism is `tracer.barrier(n)`: a value produced *inside* one invoke is not visible to a sibling invoke until the two are ordered with a barrier.

## Setup

`TransformersModel` is the primary HuggingFace class in nnsight 0.8 (the older `LanguageModel` still works but is deprecated). It implements batching, so multiple prompts can share one forward pass.

In [1]:
from nnsight.modeling.transformers import TransformersModel

model = TransformersModel("openai-community/gpt2", device_map="auto", dispatch=True)

/home/localjadenfk/miniconda3/envs/ndif2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Invokers

When you pass input directly to `model.trace(x)`, an implicit invoke is created over the whole batch. For multiple prompts, open explicit `tracer.invoke(...)` blocks — each contributes its rows to the batch, and each block's `.output` carries only that invoke's row(s).

In [2]:
with model.trace() as tracer:

    with tracer.invoke("The Eiffel Tower is in the city of"):
        logits1 = model.lm_head.output.save()

    with tracer.invoke("The Colosseum is in the city of"):
        logits2 = model.lm_head.output.save()

print(f"Prompt 1: {model.tokenizer.decode(logits1[0, -1].argmax(dim=-1))}")
print(f"Prompt 2: {model.tokenizer.decode(logits2[0, -1].argmax(dim=-1))}")

Prompt 1:  Paris
Prompt 2:  P


<details class="admonition note">
<summary>How invokers really run</summary>

Each invoke's body runs in its own **greenlet worker** (cooperative, single-threaded). Workers do **not** run strictly one-after-another: they all start together and resume in the order the model reaches what each asked for. That is what makes them a batch rather than a sequence. Within a single invoke, you must still access modules in forward-pass order, or you hit `OutOfOrderError`.

</details>

You can also batch several prompts inside a single invoke by passing a list. The saved activation then has one row per prompt:

In [3]:
with model.trace() as tracer:

    with tracer.invoke(["The Eiffel Tower is in the city of", "The Colosseum is in the city of"]):
        logits = model.lm_head.output.save()

print(f"Prompt 1: {model.tokenizer.decode(logits[0, -1].argmax(dim=-1))}")
print(f"Prompt 2: {model.tokenizer.decode(logits[1, -1].argmax(dim=-1))}")

Prompt 1:  Paris
Prompt 2:  P


## Values from the enclosing scope

A value bound **before or around** the invokes flows into every invoke automatically — no synchronization needed, because it is already materialized when the workers start. This is the easy case: the shared value did not come from another invoke's activation.

In [4]:
import torch

steer = torch.zeros(768)  # defined in the enclosing scope

with model.trace() as tracer:

    with tracer.invoke("The Eiffel Tower is in the city of"):
        a = model.lm_head.output.save()

    with tracer.invoke("The Colosseum is in the city of"):
        # `steer` came from outside the invokes, so it is visible with no barrier
        model.transformer.h[5].output[:, -1, :] += steer
        b = model.lm_head.output.save()

print(f"Shapes: {a.shape}, {b.shape}")

Shapes: torch.Size([1, 10, 50257]), torch.Size([1, 10, 50257])


## Cross-prompt transfer needs a barrier

The interesting case is transferring a value one invoke **produces from an activation** into another invoke. Because all workers start together, the consumer would run before the producer has bound the value — a `NameError`. `tracer.barrier(n)` fixes the ordering: everything written *above* a barrier happens before anything written *below* one.

Create it with `tracer.barrier(n)`, where `n` is the number of invokes that call `barrier()`. Here the first prompt's embeddings are transferred onto a second prompt made only of underscores — which then generates as if it were the first prompt.

In [5]:
with model.trace() as tracer:

    barrier = tracer.barrier(2)  # two participating invokes

    with tracer.invoke("The Eiffel Tower is in the city of"):
        embeddings = model.transformer.wte.output
        barrier()  # signal: embeddings have been read

    with tracer.invoke("_ _ _ _ _ _ _ _ _"):
        barrier()  # wait until invoke 1 has read its embeddings
        model.transformer.wte.output = embeddings
        logits = model.lm_head.output.save()

print(f"Prediction from transferred embeddings: {model.tokenizer.decode(logits[0, -1].argmax(dim=-1))}")

Prediction from transferred embeddings:  Paris


<details class="admonition warning">
<summary>When is a barrier required?</summary>

A barrier is needed whenever a later invoke **uses a value an earlier invoke produced from a module** — regardless of whether the two invokes touch the same module or different ones. If the shared value came from the enclosing scope (as with `steer` above), no barrier is needed. `n` must equal the number of invokes that actually call `barrier()`; a non-participating invoke is not counted.

</details>

## Activation patching

A practical application: paste an activation from a **clean** run into a **corrupt** run to measure that component's causal contribution. If the corrupt run then predicts the clean answer, the activation carried the deciding information.

Here both prompts are ten tokens long, so we can patch the residual stream at the differing **subject** positions from the Eiffel Tower prompt into the Colosseum prompt. A third, unpatched invoke gives a baseline in the same forward pass. Block outputs are plain tensors in current `transformers`, so we index the tensor directly (no tuple `[0]`).

In [6]:
clean = "The Eiffel Tower is in the city of"   # next token: " Paris"
corrupt = "The Colosseum is in the city of"     # next token: " Rome"

paris = model.tokenizer.encode(" Paris")[0]
rome = model.tokenizer.encode(" Rome")[0]

LAYER = 0
SUBJECT = slice(1, 5)  # the subject tokens that differ between the two prompts

with model.trace() as tracer:

    barrier = tracer.barrier(2)

    with tracer.invoke(clean):
        clean_hs = model.transformer.h[LAYER].output
        barrier()  # signal: clean_hs has been read

    with tracer.invoke(corrupt):
        barrier()  # wait until clean_hs is materialized
        hs = model.transformer.h[LAYER].output
        hs[:, SUBJECT, :] = clean_hs[:, SUBJECT, :]
        model.transformer.h[LAYER].output = hs
        patched_logits = model.lm_head.output[:, -1, :].save()

    with tracer.invoke(corrupt):  # no barrier() -> not a participant
        baseline_logits = model.lm_head.output[:, -1, :].save()

pb, pp = baseline_logits.softmax(-1), patched_logits.softmax(-1)
print(f"baseline corrupt: P(Paris)={pb[0, paris]:.3f} P(Rome)={pb[0, rome]:.3f}")
print(f"patched  corrupt: P(Paris)={pp[0, paris]:.3f} P(Rome)={pp[0, rome]:.3f}")

baseline corrupt: P(Paris)=0.003 P(Rome)=0.014
patched  corrupt: P(Paris)=0.064 P(Rome)=0.006


Patching the subject token at an early layer pushes the corrupt run toward the clean answer — the city information is read out early, at the subject position. Patch a late layer or the final position instead and the effect nearly vanishes; the position and layer you choose *are* the question you are asking.